<a href="https://colab.research.google.com/github/SakshiKhatiwada/Python/blob/main/Langchain/Tools_in_LangChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
!pip install langchain langchain_core langchain_community pydantic duckduckgo-search

# Built-in Tool: DuckDuckGoSearch

In [28]:
!pip install ddgs

In [29]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()


results = search_tool.invoke("ipl news")

print(results)

Get the latest updates on IPL T20 matches live score, points table, schedule, and auction players list. Discover real-time updates about today ... Home Tags IPL News ... Tag: IPL News ... 2008 To 2022: List Of All Orange Cap Winners In IPL And Their Runs Latest IPL News Today: Everything you must know about 2023 IPL with latest news , updates, match previews, live score , predictions, teams and players ... IPL 2025: Latest News and Trends ... The IPL auction became one of the most unpredictable in its history because of surprise player movements and ... Home IPL news ... Tag Archive for: IPL news ... Injured Delhi Daredevils opener Virender Sehwag will not play Saturday ’ s IPL match against ...


# Build-in Tool: Shell Tool

In [30]:
!pip install langchain_experimental

In [31]:
from langchain_community.tools import ShellTool

shell_tool = ShellTool()

# results = shell_tool.invoke("whoami") # these are shell commands
results = shell_tool.invoke('ls')

print(results)

Executing command:
 ls
sample_data



/usr/local/lib/python3.12/dist-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


# Custom Tools

In [32]:
from langchain_core.tools import tool

In [33]:
# Step 1 - create a function

def multiply(a,b):
  """Multiply two numbers"""
  return a*b

In [34]:
# Step 2 - Add Type Hints

def multiply(a:int, b:int) -> int:
  """Multiply two numbers"""
  return a*b

In [35]:
# Step 3 - Add Tool Decorator

@tool
def multiply(a:int, b:int) -> int:
  """Multiply two numbers"""
  return a*b

In [36]:
result = multiply.invoke({"a": 3, "b":5})
print(result)

15


In [37]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply two numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [38]:
# checking
print(search_tool.name)
print(search_tool.description)
print(search_tool.args)

duckduckgo_search
A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.
{'query': {'description': 'search query to look up', 'title': 'Query', 'type': 'string'}}


In [39]:
multiply.args_schema.model_json_schema() # llm see this

{'description': 'Multiply two numbers',
 'properties': {'a': {'title': 'A', 'type': 'integer'},
  'b': {'title': 'B', 'type': 'integer'}},
 'required': ['a', 'b'],
 'title': 'multiply',
 'type': 'object'}

# Method 2: Using StructuredTool

In [40]:
from langchain.tools import StructuredTool
from pydantic import BaseModel, Field

In [41]:
class MultiplyInput(BaseModel):
  a: int = Field(required=True, description="The First number to add")
  b: int = Field(required=True, description="The second number to add")

In [42]:
def multiply_func(a:int, b:int) -> int:
  return a*b

In [43]:
multiply_tool = StructuredTool.from_function(
    func=multiply_func, # our function
    name="Multiple",
    description="Multiply two numbers",
    args_schema=MultiplyInput # our pydantic class
)

In [44]:
result = multiply_tool.invoke({"a": 3, "b":5})
print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

15
Multiple
Multiply two numbers
{'a': {'description': 'The First number to add', 'required': True, 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'required': True, 'title': 'B', 'type': 'integer'}}


# Method 3: Using BaseTool Class

> We can create async version too, which isn't available in other methods

In [45]:
from langchain.tools import BaseTool
from typing import Type

In [47]:
# arg schema using pydantic
class MultiplyInput(BaseModel):
  a: int = Field(required=True, description="The First number to add")
  b: int = Field(required=True, description="The second number to add")

In [49]:
class MultiplyTool(BaseTool):
  name: str = 'multiply'
  description: str = "Multiply two numbers"
  args_schema: Type[BaseModel] = MultiplyInput

  def _run(self, a: int, b: int)-> int: # this method name can only be "_run"
    return a * b

In [50]:
multiply_tool = MultiplyTool()

In [51]:
result = multiply_tool.invoke({"a":3, "b":3})

print(result)
print(multiply_tool.name)

9
multiply


In [52]:
print(multiply_tool.args_schema)

<class '__main__.MultiplyInput'>


# Toolkit

In [ ]:
from langchain_core.tools import tool

# custom tools
@tool
def multiply(a:int, b:int) -> int:
  """Multiply two numbers"""
  return a*b

@tool
def add(a:int, b:int) -> int:
  """Add two numbers"""
  return a+b

In [ ]:
class MathToolkit:
  def get_tools(self):
    return [add, multiply]

In [ ]:
toolkit = MathToolKit()
tools = toolkit.get_tools()
print(type(tools))

for tool in tools:
  print(tool.name "=> ", tool.description)